# Loan Status

**Dataset:** loan (binary classification)

**Monotone features:**
- Increasing: `feature_1`, `feature_4`
- Decreasing: `feature_0`, `feature_2`, `feature_3`

**Budget:** 25 trials per flavor × 4 flavors (smaller dataset). Final eval: 5 seeds, best-3-of-5.

In [ ]:
import json
from pathlib import Path

from benchmarks.datasets.download import default_dest
from benchmarks.datasets.registry import load
from benchmarks._common.search import search, final_eval, flavor_name

DATASET = "loan"
BACKEND = "torch"
N_TRIALS = 25          # per (dataset, flavor); see header for per-dataset budget
EPOCHS = 50
OUT = Path("../results/phase2")
OUT.mkdir(parents=True, exist_ok=True)

bundle = load(DATASET, data_dir=default_dest())
flavors = [("switch", False), ("switch", True), ("absolute", False), ("absolute", True)]

for mode, residual in flavors:
    study = search(bundle, mode=mode, residual=residual, backend=BACKEND,
                   n_trials=N_TRIALS, epochs=EPOCHS,
                   storage=f"sqlite:///{OUT}/{DATASET}-{flavor_name(mode, residual)}.db")
    agg = final_eval(bundle, study.best_params, mode=mode, residual=residual,
                     backend=BACKEND, epochs=EPOCHS, seeds=range(5), top_k=3)
    rec = {
        "dataset": DATASET, "flavor": study.flavor,
        "best_params": study.best_params, "val_best": study.best_value,
        "test_metric": agg.metric, "test_mean": agg.mean, "test_std": agg.std,
        "n_seeds": agg.n_seeds, "n_selected": agg.n_selected,
    }
    (OUT / f"{DATASET}-{study.flavor}.json").write_text(json.dumps(rec, indent=2))
    print(study.flavor, agg.mean, "±", agg.std)